# cuPAMI quick start

Select the jhub kernel and run the cells below.


In [1]:
import cupami as cuPAMI

## Generate data

In [2]:
generator = cuPAMI.TransactionalDatabase(
    transactions=10000,
    items=50,
    avg_length=6,
    seed=42,
)
transactions = generator.create()

## Apriori

In [3]:
apriori = cuPAMI.cuApriori(transactions, minSup=0.01)
apriori.mine()
print("Patterns:", len(apriori.getPatterns()))
print("Loading (s):", apriori.getLoadingTime())
print("Mining (s):", apriori.getMiningTime())
print("Total (s):", apriori.getTotalRuntime())

Patterns: 1275
Loading (s): 0.042862712987698615
Mining (s): 0.7180253909900784
Total (s): 0.7608881039777771


## Eclat

In [4]:
eclat = cuPAMI.cuEclat(transactions, minSup=0.01)
eclat.mine()
print("Patterns:", len(eclat.getPatterns()))
print("Loading (s):", eclat.getLoadingTime())
print("Mining (s):", eclat.getMiningTime())
print("Total (s):", eclat.getTotalRuntime())

Patterns: 1275
Loading (s): 0.012869080004747957
Mining (s): 0.2544529799488373
Total (s): 0.26732205995358527


## FP-growth

In [5]:
fpgrowth = cuPAMI.cuFPGrowth(transactions, minSup=0.01)
fpgrowth.mine()
print("Patterns:", len(fpgrowth.getPatterns()))
print("Loading (s):", fpgrowth.getLoadingTime())
print("Mining (s):", fpgrowth.getMiningTime())
print("Total (s):", fpgrowth.getTotalRuntime())

Patterns: 1275
Loading (s): 0.049934530979953706
Mining (s): 0.6089468710124493
Total (s): 0.658881401992403


## View some patterns

Each entry contains a pattern and its support count.

In [6]:
list(apriori.getPatterns().items())[:5]

[('1', 1208), ('10', 1151), ('11', 1201), ('12', 1176), ('13', 1148)]

## Temporal data: unique timestamps

In [7]:
unique = cuPAMI.TemporalDatabase(
    transactions=8,
    items=10,
    avg_length=3,
    seed=42,
    repeatAllowed=False,
    missingAllowed=False,
    mergeTimestamps=False,
).create()

for row in unique:
    print(*row)

1 2 4 5 9
2 1 2 7 9
3 9
4 4 7 9
5 1 3 7
6 2 6 9
7 1 5 6
8 2 5 7 10


## Temporal data: repeated timestamps

In [8]:
separate = cuPAMI.TemporalDatabase(
    transactions=8,
    items=10,
    avg_length=3,
    seed=42,
    repeatAllowed=True,
    missingAllowed=False,
    mergeTimestamps=False,
).create()

for row in separate:
    print(*row)

1 2 4 5 9
1 1 2 7 9
1 1 10
2 4 5 7 8
2 2 3 6 7 10
3 2 3 7 9 10
4 1 2 7 9
4 1 2 4 6


## Temporal data: merged timestamps

In [9]:
merged = cuPAMI.TemporalDatabase(
    transactions=8,
    items=10,
    avg_length=3,
    seed=42,
    repeatAllowed=True,
    missingAllowed=False,
    mergeTimestamps=True,
).create()

for row in merged:
    print(*row)

1 1 2 4 5 7 9 10
2 2 3 4 5 6 7 8 10
3 2 3 7 9 10
4 1 2 4 6 7 9


## Temporal data: missing timestamps

In [10]:
gaps = cuPAMI.TemporalDatabase(
    transactions=8,
    items=10,
    avg_length=3,
    seed=42,
    repeatAllowed=False,
    missingAllowed=True,
    mergeTimestamps=False,
).create()

for row in gaps:
    print(*row)

1 2 4 5 9
2 1 2 7 9
3 1 10
5 4 5 7 8
6 2 3 6 7 10
8 2 3 7 9 10
10 1 2 7 9
11 1 2 4 6


## Fuzzy CSV mining

In [11]:
file = "https://u-aizu.ac.jp/~udayrage/datasets/fuzzyDatabases/Fuzzy_T10I4D100K.csv"
fuzzy = cuPAMI.cuFFIMiner(file, minSup=0.005)
fuzzy.mine()
print("Patterns:", len(fuzzy.getPatterns()))
print("Loading (s):", fuzzy.getLoadingTime())
print("Mining (s):", fuzzy.getMiningTime())
print("Total (s):", fuzzy.getTotalRuntime())

Patterns: 787
Loading (s): 2.584315618034452
Mining (s): 1.411760910996236
Total (s): 3.996076529030688


## Multiple fuzzy regions

In [ ]:
from urllib.parse import urlparse
from urllib.request import urlretrieve

# SBT requires a local file. Download the sample if it is a URL.
local_fuzzy_file = file
if urlparse(str(file)).scheme in ("http", "https"):
    local_fuzzy_file, _ = urlretrieve(file, "fuzzy.csv")

multiple = cuPAMI.cuMFFIMiner(local_fuzzy_file, minSup=0.005)
multiple.mine()
print("Patterns:", len(multiple.getPatterns()))
print("Loading (s):", multiple.getLoadingTime())
print("Mining (s):", multiple.getMiningTime())
print("Total (s):", multiple.getTotalRuntime())

## Spatial patterns

Transactions and neighbours use the same CSV format as PAMI.

In [2]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as folder:
    folder = Path(folder)
    transactions_file = folder / "transactions.csv"
    neighbours_file = folder / "neighbours.csv"
    transactions_file.write_text("a,b,c\na,b\nb,c\n")
    neighbours_file.write_text("a,b\nb,a,c\nc,b\n")
    spatial = cuPAMI.cuSpatialEclat(transactions_file, neighbours_file, minSup=2)
    spatial.mine()
    print("Patterns:", len(spatial.getPatterns()))
    display(spatial.getPatternsAsDataFrame())

Patterns: 5


,Patterns,Support
0,a,2
1,b,3
2,c,2
3,a\tb,2
4,b\tc,2


## Binary transactions

In [ ]:
binary_generator = cuPAMI.BinaryTransactionalDatabase(
    transactions=8,
    items=10,
    avg_length=3,
    seed=42,
)
binary_transactions = binary_generator.create()

assert len(binary_transactions) == 8
assert all(row == sorted(set(row)) for row in binary_transactions)
for row in binary_transactions:
    print(*row)

2 4 5 9
1 2 7 9
9
4 7 9
1 3 7
2 6 9
1 5 6
2 5 7 10


## Quantitative transactions

In [ ]:
quantitative_generator = cuPAMI.QuantitativeTransactionalDatabase(
    transactions=8,
    items=10,
    avg_length=3,
    seed=42,
    min_quantity=1,
    max_quantity=10,
)
quantitative_transactions = quantitative_generator.create()

assert len(quantitative_transactions) == 8
assert all(1 <= quantity <= 10
           for row in quantitative_transactions for quantity in row.values())
for row in quantitative_transactions:
    print(row)

{2: 2, 4: 9, 5: 2, 9: 10}
{1: 9, 4: 4, 10: 9}
{1: 7, 3: 6, 5: 5}
{6: 2}
{5: 1, 6: 8}
{2: 10, 5: 6, 7: 10}
{4: 2, 5: 4}
{3: 4, 6: 5, 7: 2, 8: 10, 10: 3}


## Convert datasets: CSV, BIN, Parquet, and TSV

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as folder:
    folder = Path(folder)
    binary_generator.save(folder / "transactions.csv")
    for source, output in (
        ("transactions.csv", "transactions.bin"),
        ("transactions.bin", "transactions.parquet"),
        ("transactions.parquet", "transactions.tsv"),
        ("transactions.tsv", "roundtrip.csv"),
    ):
        result = cuPAMI.convertFile(folder / source, folder / output, dataType="binary")
        assert result["transactions"] == 8
        print(source, "->", output, result)
    assert (folder / "transactions.csv").read_text() == (folder / "roundtrip.csv").read_text()

    quantitative_generator.save(folder / "quantities.csv")
    for source, output in (
        ("quantities.csv", "quantities.bin"),
        ("quantities.bin", "quantities.parquet"),
        ("quantities.parquet", "quantities.tsv"),
        ("quantities.tsv", "quantities-roundtrip.csv"),
    ):
        result = cuPAMI.convertFile(folder / source, folder / output, dataType="quantitative")
        assert result["transactions"] == 8
        print(source, "->", output, result)
    assert (folder / "quantities.csv").read_text() == (folder / "quantities-roundtrip.csv").read_text()

transactions.csv -> transactions.bin {'transactions': 8, 'item_occurrences': 25, 'max_item_id': 10, 'format': 'bin'}


transactions.bin -> transactions.parquet {'transactions': 8, 'item_occurrences': 25, 'max_item_id': 10, 'format': 'parquet'}
transactions.parquet -> transactions.tsv {'transactions': 8, 'item_occurrences': 25, 'max_item_id': 10, 'format': 'tsv'}
transactions.tsv -> roundtrip.csv {'transactions': 8, 'item_occurrences': 25, 'max_item_id': 10, 'format': 'csv'}
quantities.csv -> quantities.bin {'transactions': 8, 'item_occurrences': 23, 'max_item_id': 10, 'format': 'bin'}
quantities.bin -> quantities.parquet {'transactions': 8, 'item_occurrences': 23, 'max_item_id': 10, 'format': 'parquet'}
quantities.parquet -> quantities.tsv {'transactions': 8, 'item_occurrences': 23, 'max_item_id': 10, 'format': 'tsv'}
quantities.tsv -> quantities-roundtrip.csv {'transactions': 8, 'item_occurrences': 23, 'max_item_id': 10, 'format': 'csv'}


## Partial-periodic Apriori

In [ ]:
partial_rows = [
    [1, "a", "b"],
    [3, "a", "b"],
    [5, "a"],
    [10, "a", "b"],
]
partial_apriori = cuPAMI.cuPPPApriori(partial_rows, minPS=1, period=2)
partial_apriori.mine()
assert partial_apriori.getPatterns() == {"a": 2, "b": 1, "a\tb": 1}
print("Patterns:", len(partial_apriori.getPatterns()))
print("Loading (s):", partial_apriori.getLoadingTime())
print("Mining (s):", partial_apriori.getMiningTime())
print("Total (s):", partial_apriori.getTotalRuntime())
display(partial_apriori.getPatternsAsDataFrame())

Patterns: 3
Loading (s): 8.139992132782936e-05
Mining (s): 0.9210837921127677
Total (s): 0.9211651920340955


,Patterns,periodicSupport
0,a,2
1,b,1
2,a\tb,1


## Partial-periodic Eclat

In [ ]:
partial_eclat = cuPAMI.cuPPPEclat(partial_rows, minPS=1, period=2)
partial_eclat.mine()
assert partial_eclat.getPatterns() == partial_apriori.getPatterns()
print("Patterns:", len(partial_eclat.getPatterns()))
print("Loading (s):", partial_eclat.getLoadingTime())
print("Mining (s):", partial_eclat.getMiningTime())
print("Total (s):", partial_eclat.getTotalRuntime())
display(partial_eclat.getPatternsAsDataFrame())

Patterns: 3
Loading (s): 6.274017505347729e-05
Mining (s): 0.000860084081068635
Total (s): 0.0009228242561221123


,Patterns,periodicSupport
0,a,2
1,b,1
2,a\tb,1
